<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F12_conversational_analytics_and_knowledge_catalog.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.

# 12 · Self-Documenting Views, Knowledge Catalog & Stateful BigQuery `DataAgent`

Every `scale-forecasting` registry ships **5 self-documenting BigQuery tables** and **11 self-documenting analytical SQL views** (`5` core operational views + `6` diagnostic and S&OP views) with table, view, and column descriptions applied automatically at creation and backfilled idempotently onto existing registries.

On top of that structured SQL foundation, `scale_forecasting.bq_agent` synchronizes a **44-term Knowledge Catalog Business Glossary** (`WAPE`, `MASE`, `MSIS`, `FVA`, `Trigg's Tracking Signal`, `SBC`, `THieF`, `snapshot_millis`, etc.), optional **Knowledge Catalog** `DataProfile` scans, **32 verified Golden SQL exemplars**, and a **stateful BigQuery Conversational Analytics `DataAgent`** (`sf-registry-<dataset>`) scoped **strictly** to the registry's 5 tables, 11 views, and verified `source_table`s from `run_registry`.

Every conversational cell in this notebook uses the non-raising **`reg.probe_bq_agent()` check-and-run pattern**: pure BigQuery SQL views always execute, while stateful `DataAgent` calls run automatically when the Conversational Analytics API is enabled and print exact setup guidance if not.

### At a glance
| Dimension | Details |
| :--- | :--- |
| **Learning outcomes** | Probe Conversational Analytics & **Knowledge Catalog** readiness (`reg.probe_bq_agent()`), audit `100%` table/view/column description coverage in `INFORMATION_SCHEMA`, query all 6 diagnostic & S&OP SQL views (`v_executive_sop_summary`, `v_fva_waterfall`, `v_horizon_decay`, `v_bias_and_exceptions`, `v_anomalies`, `v_feature_attributions`), synchronize the **44-term Knowledge Catalog Business Glossary** & strict `BQAgentScope` (`reg.ensure_bq_agent()`), run a **4-turn stateful conversation** (`conversation_id`), pin historical `run_id`s (`Forecaster.from_run_id().ask()`), and invoke the MCP `ask_forecast_registry` tool. |
| **Prerequisites** | Deployed `scale-forecasting` Google Cloud project (`SF_*` env vars) + `scale-forecasting[gcp]`. Optional: `geminidataanalytics.googleapis.com` & `dataplex.googleapis.com` enabled for live `DataAgent` turns (Terraform enables these by default). |
| **Estimated time** | ~2–4 minutes. |
| **Estimated run cost** | **Estimated `~\$0.01–\$0.05` USD** (BigQuery `INFORMATION_SCHEMA` + view queries and serverless Conversational Analytics turns over existing registry data). |
| **Ongoing cost** | **Zero compute when idle** — BigQuery views, the **Knowledge Catalog** Business Glossary, and the registered `DataAgent` metadata resource incur `\$0` idle compute charges. |

```mermaid
flowchart TB
    subgraph SQL["Layer 1: Self-Documenting BigQuery Registry (Always Available)"]
        T5["5 Registry Tables + Discovered Source Tables<br/>100% Table & Column Descriptions + snapshot_millis Lineage"]
        V11["11 Analytical SQL Views (5 Core + 6 Diagnostic & S&OP)<br/>v_executive_sop_summary · v_fva_waterfall · v_horizon_decay<br/>v_bias_and_exceptions · v_anomalies · v_feature_attributions"]
    end
    subgraph KC["Layer 2: Knowledge Catalog Governance"]
        G44["44-Term Forecasting & S&OP Business Glossary<br/>6 Categories · WAPE · MASE · FVA · Tracking Signal · SBC · THieF"]
        DP["Optional Knowledge Catalog DataProfile & Doc Scans<br/>ensure_knowledge_catalog_scans()"]
    end
    subgraph Agent["Layer 3: Check-and-Run Stateful Conversational Analytics"]
        PR["reg.probe_bq_agent()<br/>Non-Raising API & IAM Readiness Guard"]
        DA["Registered BigQuery DataAgent (sf-registry-{dataset})<br/>Strict Scope + 32 Golden SQL Queries + Latest COMPLETED Default"]
        SUR["4 Consumption Surfaces<br/>BigQuery Studio · reg.ask() / f.ask() · CLI · MCP ask_forecast_registry"]
    end
    SQL --> KC --> PR --> DA --> SUR
```

## Act 1 · Cloud Bootstrap, Deployment Resolution & Check-and-Run Probe (`reg.probe_bq_agent()`)

On **Colab Enterprise** or **Vertex AI Workbench**, the bootstrap cell installs the locked `scale-forecasting[gcp]` dependency set (`uv.lock`) when running in a fresh kernel; in a local clone with `.venv`, it is an instant no-op.

In [1]:
# Cloud bootstrap: clone + install the LOCKED dependency set (uv.lock) so
# `import scale_forecasting` resolves against the exact versions every other surface runs.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")
EXTRAS = ['gcp']

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    uv = shutil.which("uv")
    if uv is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or "uv"
    reqs = os.path.abspath(os.path.join(REPO_DIR, "colab-requirements.txt"))
    extra_flags = [f for e in EXTRAS for f in ("--extra", e)]
    subprocess.run(
        [uv, "export", "--frozen", "--no-emit-project", "--no-hashes", "--no-dev", *extra_flags, "-o", reqs],
        cwd=REPO_DIR,
        check=True,
    )
    target = os.path.abspath(os.path.join(REPO_DIR, ".colab-deps"))
    subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "--target", target, "-r", reqs],
        check=True,
    )
    sys.path.insert(0, target)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    print("installed scale_forecasting from", REPO_DIR)
else:
    print("scale_forecasting already importable — skipping bootstrap")

scale_forecasting already importable — skipping bootstrap


### Non-Raising Readiness Probe (`reg.probe_bq_agent()`)

`reg.probe_bq_agent()` (and CLI `python -m scale_forecasting.bq_agent --probe` / MCP `ask_forecast_registry(mode="probe")`) inspects whether BigQuery Conversational Analytics (`geminidataanalytics.googleapis.com`) and **Knowledge Catalog** (`dataplex.googleapis.com`) are enabled and reachable **without raising**. If an environment has not opted in yet, `agent_status.setup_hint` prints the exact one-line enablement command while allowing pure-SQL notebook cells to run uninterrupted.

In [2]:
import json
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module=r"google\..*")

import pandas as pd
from google.cloud import bigquery

from scale_forecasting import Forecaster, Registry, RunConfig, probe_bq_agent_access
from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
reg = Registry(settings=settings)

# Ensure all 5 tables + 11 views exist and backfill any missing descriptions idempotently:
reg.init()

agent_status = reg.probe_bq_agent()
print("Live Registry BQ Agent Probe:")
print(json.dumps(agent_status.to_dict(), indent=2))

# Demonstrate graceful fallback behavior when unconfigured:
unconfigured_probe = probe_bq_agent_access(project_id="", dataset_id="scale_forecasting")
print(f"\nFallback preview when unconfigured -> ready={unconfigured_probe.ready} | hint: {unconfigured_probe.setup_hint}")

Live Registry BQ Agent Probe:
{
  "ready": true,
  "conversational_api_ready": true,
  "knowledge_catalog_ready": true,
  "agent_registered": true,
  "project_id": "statmike-scale-forecasting",
  "dataset_id": "scale_forecasting",
  "location": "global",
  "agent_id": "sf-registry-scale-forecasting",
  "agent_name": "projects/statmike-scale-forecasting/locations/global/dataAgents/sf-registry-scale-forecasting",
  "missing_apis": [],
  "reason": "BigQuery Conversational Analytics API ready (registered); Knowledge Catalog API ready.",
  "setup_hint": ""
}



Fallback preview when unconfigured -> ready=True | hint: 


## Act 2 · Self-Documenting `INFORMATION_SCHEMA` Audit & All 6 Diagnostic SQL Views (Pure SQL)

Before touching any LLM or conversational endpoint, every table and view in the registry carries authoritative descriptions directly in BigQuery `OPTIONS(description=...)` and per-column `FIELD_PATHS` metadata (`src/scale_forecasting/registry/metadata.py`). Let's verify `100%` description coverage across all **16 canonical registry objects** (`5` tables + `11` views) and inspect the server-side diagnostic views directly.

In [3]:
from scale_forecasting.registry.ddl import TABLE_NAMES
from scale_forecasting.registry.views import VIEW_NAMES

canonical_objects = list(TABLE_NAMES) + list(VIEW_NAMES)
dataset_fqn = f"{settings.project_id}.{settings.registry_dataset_id}"

meta_sql = f"""
WITH obj_desc AS (
  SELECT table_name, TRIM(option_value, '"') AS table_description
  FROM `{dataset_fqn}.INFORMATION_SCHEMA.TABLE_OPTIONS`
  WHERE option_name = 'description'
),
col_stats AS (
  SELECT
    table_name,
    COUNT(*) AS total_columns,
    COUNTIF(description IS NOT NULL AND LENGTH(TRIM(description)) > 0) AS described_columns
  FROM `{dataset_fqn}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
  WHERE field_path = column_name
  GROUP BY table_name
)
SELECT
  t.table_name,
  t.table_type,
  c.total_columns,
  c.described_columns,
  ROUND(SAFE_DIVIDE(c.described_columns, c.total_columns) * 100, 1) AS col_desc_pct,
  SUBSTR(o.table_description, 1, 88) || '...' AS description_preview
FROM `{dataset_fqn}.INFORMATION_SCHEMA.TABLES` AS t
LEFT JOIN obj_desc AS o USING (table_name)
LEFT JOIN col_stats AS c USING (table_name)
WHERE t.table_name IN UNNEST(@names)
ORDER BY IF(t.table_type = 'VIEW', 1, 0), t.table_name
"""
job_cfg = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("names", "STRING", canonical_objects)]
)
schema_audit_df = client.query(meta_sql, job_config=job_cfg).to_dataframe()
print(
    f"Audited {len(schema_audit_df)}/{len(canonical_objects)} registry tables & views | "
    f"Mean column description coverage: {schema_audit_df['col_desc_pct'].mean():.1f}%"
)
display(schema_audit_df)

# 1. Executive S&OP Scorecard (`v_executive_sop_summary`) via SDK helper:
sop_df = reg.sop_summary_df(limit=5)
print("\n--- 1. Executive S&OP Summary (v_executive_sop_summary) ---")
display(
    sop_df[
        [
            "run_id",
            "status",
            "champion_model",
            "champion_pooled_wape",
            "fva_relative_lift_pct",
            "total_forecast_units",
            "exception_rate",
        ]
    ]
)

latest_completed_run = str(
    sop_df[sop_df["status"] == "COMPLETED"].iloc[0]["run_id"]
    if not sop_df[sop_df["status"] == "COMPLETED"].empty
    else sop_df.iloc[0]["run_id"]
)
run_param = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("rid", "STRING", latest_completed_run)]
)

# 2. Preview FVA Waterfall & Horizon Decay on the latest completed run:
fva_sql = f"""
SELECT stage_order, stage_name, model_type, ROUND(pooled_wape, 4) AS pooled_wape,
       baseline_model, ROUND(fva_relative_lift_pct, 2) AS fva_relative_lift_pct
FROM `{dataset_fqn}.v_fva_waterfall`
WHERE run_id = @rid
ORDER BY stage_order, pooled_wape ASC
LIMIT 8
"""
print(f"\n--- 2. Forecast Value Added Waterfall (v_fva_waterfall | run_id={latest_completed_run}) ---")
display(client.query(fva_sql, job_config=run_param).to_dataframe())

decay_sql = f"""
SELECT model_type, horizon_step, n_series, n_points, ROUND(pooled_wape, 4) AS pooled_wape,
       ROUND(empirical_coverage, 3) AS empirical_coverage, ROUND(mean_interval_width, 2) AS mean_width
FROM `{dataset_fqn}.v_horizon_decay`
WHERE run_id = @rid AND horizon_step IN (1, 3, 7, 14)
ORDER BY model_type, horizon_step
LIMIT 8
"""
print(f"\n--- 3. Lead-Time Horizon Decay Sample (v_horizon_decay | steps 1, 3, 7, 14) ---")
display(client.query(decay_sql, job_config=run_param).to_dataframe())

Audited 18/18 registry tables & views | Mean column description coverage: 100.0%


,table_name,table_type,total_columns,described_columns,col_desc_pct,description_preview
0,backtest_oof,BASE TABLE,16,16,100.0,Snapshot-locked out-of-fold (OOF) historical b...
1,forecast_metadata,BASE TABLE,67,67,100.0,"Per-(run_id, ts_id, model_type) fit diagnostic..."
2,forecast_predictions,BASE TABLE,14,14,100.0,Forward horizon point and probabilistic foreca...
3,run_jobs,BASE TABLE,16,16,100.0,Per-family cloud job execution trace (append-o...
4,run_registry,BASE TABLE,17,17,100.0,scale-forecasting run header ledger (append-on...
5,source_series_iceberg,BASE TABLE,5,5,100.0,Shipped univariate benchmark source panel stor...
6,source_series_native,BASE TABLE,5,5,100.0,Shipped univariate benchmark source panel stor...
7,v_anomalies,VIEW,14,14,100.0,Historical out-of-fold demand anomalies and in...
8,v_backtest_coverage,VIEW,9,9,100.0,Achieved backtest fold coverage cohort table p...
9,v_bias_and_exceptions,VIEW,17,17,100.0,"Planner exception triage queue per (run_id, ts..."



--- 1. Executive S&OP Summary (v_executive_sop_summary) ---


,run_id,status,champion_model,champion_pooled_wape,fva_relative_lift_pct,total_forecast_units,exception_rate
0,track-d1-live-agent-auto-refresh-15c1fc6a7155,COMPLETED,ensemble_mean,0.085264,10.278194,10683.148534,0.833333
1,track-p-p2-live-sim-hierarchy-3mode-01dd2c1edab7,COMPLETED,lightgbm,0.093330,34.868312,55311.330490,0.750000
2,track-p-p2-live-sharded-sim-global-0916c3ff272e,COMPLETED,holtwinters,0.138792,22.941617,16829.652332,0.916667
3,track-p-p2-live-sharded-sim-global-df186104f037,RUNNING,holtwinters,0.138792,16.299678,16829.652332,0.916667
4,track-p-p2-live-sharded-sim-global-428eaac8390c,FAILED,holtwinters,0.138792,16.299678,16829.652332,0.916667



--- 2. Forecast Value Added Waterfall (v_fva_waterfall | run_id=track-d1-live-agent-auto-refresh-15c1fc6a7155) ---


,stage_order,stage_name,model_type,pooled_wape,baseline_model,fva_relative_lift_pct
0,2,2_statistical,arima_plus@xreg,0.0924,arima_plus,2.77
1,2,2_statistical,arima_plus,0.0950,arima_plus,0.00
2,3,3_univariate_ml_dl,timesfm@univariate,0.0886,arima_plus,6.72
3,4,4_covariate_ml_dl,timesfm,0.0919,arima_plus,3.26
4,5,5_ensemble,ensemble_mean,0.0853,arima_plus,10.28
5,5,5_ensemble,ensemble_inverse_error,0.0858,arima_plus,9.72
6,5,5_ensemble,ensemble_nnls,0.0863,arima_plus,9.19
7,5,5_ensemble,ensemble_champion,0.0921,arima_plus,3.08



--- 3. Lead-Time Horizon Decay Sample (v_horizon_decay | steps 1, 3, 7, 14) ---


,model_type,horizon_step,n_series,n_points,pooled_wape,empirical_coverage,mean_width
0,arima_plus,1,6,12,0.0558,0.667,17.42
1,arima_plus,3,6,12,0.1766,0.500,20.05
2,arima_plus,7,6,12,0.1180,0.583,20.42
3,arima_plus,14,6,12,0.0842,0.500,20.43
4,arima_plus@xreg,1,6,12,0.0689,0.583,21.61
5,arima_plus@xreg,3,6,12,0.1553,0.583,24.60
6,arima_plus@xreg,7,6,12,0.0867,0.750,24.92
7,arima_plus@xreg,14,6,12,0.0678,0.583,24.95


## Act 3 · Strict Registry Scope, Knowledge Catalog Business Glossary & `DataAgent` Registration

A BigQuery dataset can contain temporary staging tables (`_sf_automl_*`) or unrelated enterprise tables. `discover_registry_scope()` restricts the agent's `publishedContext` **exclusively** to:
1. The **5 canonical registry tables** (`run_registry`, `forecast_predictions`, `forecast_metrics`, `backtest_oof`, `ensemble_weights`).
2. The **11 analytical SQL views**.
3. Verified **`source_table`s** recorded in `run_registry` (paired with `snapshot_millis` time-travel awareness).

When `agent_status.ready` is `True`, `reg.ensure_bq_agent(sync_glossary=True)` idempotently synchronizes all **44 Knowledge Catalog Business Glossary terms** across 6 categories and registers or refreshes `sf-registry-<dataset>` with **32 Golden SQL exemplars**.

In [4]:
from scale_forecasting.bq_agent import (
    GLOSSARY_TERMS,
    build_golden_queries,
    discover_registry_scope,
)

scope = discover_registry_scope(
    settings.project_id,
    settings.registry_dataset_id,
    client=client,
)
staging_leaks = [obj.table_id for obj in scope.all_objects if "_sf_automl_" in obj.table_id]
print(
    f"Discovered Strict Scope: {len(scope.registry_tables)} registry tables, "
    f"{len(scope.registry_views)} analytical views, "
    f"{len(scope.source_tables)} verified source tables "
    f"(staging leaks={len(staging_leaks)})"
)

glossary_preview = pd.DataFrame(
    [
        {
            "term_id": t.term_id,
            "display_name": t.display_name,
            "labels": ", ".join(t.labels),
            "n_linked_objects": len(t.linked_objects),
        }
        for t in GLOSSARY_TERMS[:8]
    ]
)
golden_queries = build_golden_queries(settings.project_id, settings.registry_dataset_id)
print(
    f"Knowledge Catalog Business Glossary: {len(GLOSSARY_TERMS)} terms | "
    f"Golden SQL Exemplars: {len(golden_queries)}"
)
display(glossary_preview)

if agent_status.ready:
    receipt = reg.ensure_bq_agent(sync_glossary=True, run_scans=False)
    action_label = "created" if receipt.created else "updated"
    print(
        f"\nProvisioned DataAgent: {receipt.agent_name} ({action_label})\n"
        f"Knowledge Catalog Glossary: {receipt.glossary_name}\n"
        f"BigQuery Studio URL: {receipt.console_url}"
    )
else:
    print(f"\nSkipping live DataAgent provisioning: {agent_status.setup_hint}")

Discovered Strict Scope: 5 registry tables, 11 analytical views, 21 verified source tables (staging leaks=0)
Knowledge Catalog Business Glossary: 44 terms | Golden SQL Exemplars: 32


,term_id,display_name,labels,n_linked_objects
0,wape,WAPE,"accuracy, metric",8
1,mase,MASE,"accuracy, metric",2
2,smape,sMAPE,"accuracy, metric",1
3,rmsse,RMSSE,"accuracy, metric",1
4,mae,MAE,"accuracy, metric",1
5,rmse,RMSE,"accuracy, metric",1
6,winkler-score,Winkler Score,"uncertainty, metric",1
7,pinball-loss,Pinball Loss,"uncertainty, metric",1



Provisioned DataAgent: projects/statmike-scale-forecasting/locations/global/dataAgents/sf-registry-scale-forecasting (updated)
Knowledge Catalog Glossary: projects/statmike-scale-forecasting/locations/global/glossaries/sf-glossary-scale-forecasting
BigQuery Studio URL: https://console.cloud.google.com/bigquery?project=statmike-scale-forecasting


## Act 4 · Multi-Turn Stateful Conversation, Historical `run_id` Pinning & Cross-Run Rollups

Because `sf-registry-<dataset>` is registered as a **stateful BigQuery Conversational Analytics `DataAgent`**, passing `conversation_id=turn1.conversation_id` preserves server-side conversation history across follow-up turns (`"that same run"`, `"the winning model"`) without repeating filters:
- **Default (`run_id=None` / `"latest"`):** Resolves to the newest `COMPLETED` run in `v_run_summary`.
- **Historical Run Pinning (`Forecaster.from_run_id(rid).ask(...)`):** Scopes questions to a specific archived run.
- **Cross-Run Comparison (`run_id="all"`):** Compares trends across multiple runs in `v_executive_sop_summary` or `v_anomalies`.

In [5]:
if agent_status.ready:
    # Turn 1: Champion model & accuracy on the latest COMPLETED run
    t1 = reg.ask(
        "Which model won on the latest completed run and what were its pooled WAPE and pooled MAE?"
    )
    print(f"[Turn 1 | run_id={t1.run_id} | conversation={t1.conversation_id}]\n{t1.answer}")
    if t1.sql:
        print(f"Generated SQL:\n{t1.sql}\n")

    # Turn 2: Stateful follow-up on FVA Waterfall (reuses t1.conversation_id)
    t2 = reg.ask(
        "Show the Forecast Value Added (FVA) waterfall stages and relative lift % for that same run.",
        conversation_id=t1.conversation_id,
    )
    print(f"[Turn 2 | FVA Waterfall follow-up | rows={len(t2.df)}]\n{t2.answer}")
    if not t2.df.empty:
        display(t2.df.head(6))

    # Turn 3: Stateful follow-up on Horizon Decay
    t3 = reg.ask(
        "How does pooled_wape change across horizon_step 1, 7, and 14 for the winning model on that run?",
        conversation_id=t1.conversation_id,
    )
    print(f"\n[Turn 3 | Horizon Decay follow-up | rows={len(t3.df)}]\n{t3.answer}")

    # Turn 4: Stateful follow-up on Planner Bias & Tracking Signal Exceptions
    t4 = reg.ask(
        "How many series are flagged with is_exception = TRUE in v_bias_and_exceptions for the winning model on that run?",
        conversation_id=t1.conversation_id,
    )
    print(f"\n[Turn 4 | Bias & Exception Triage follow-up | rows={len(t4.df)}]\n{t4.answer}")

    # Historical run pinning (`Forecaster.from_run_id(...).ask(...)`)
    hist_fc = Forecaster.from_run_id(
        "track-p-p2-live-sim-hierarchy-3mode-01dd2c1edab7", settings=settings
    )
    hist_ans = hist_fc.ask(
        "Which model won on this historical run in v_model_leaderboard_comparable and what was its pooled_wape?"
    )
    print(f"\n[Historical Run Pinning | run_id={hist_ans.run_id}]\n{hist_ans.answer}")

    # Cross-run S&OP executive comparison (`run_id='all'`)
    cross_run = reg.ask(
        "List the 5 most recent completed runs from v_executive_sop_summary with champion_model, champion_pooled_wape, and total_forecast_units.",
        run_id="all",
    )
    print(f"\n[Cross-Run S&OP Comparison | run_id={cross_run.run_id} | rows={len(cross_run.df)}]\n{cross_run.answer}")
    if not cross_run.df.empty:
        display(cross_run.df.head(5))
else:
    print(f"Skipping live multi-turn Q&A: {agent_status.setup_hint}")

[Turn 1 | run_id=track-d1-live-agent-auto-refresh-15c1fc6a7155 | conversation=sf-conv-5929a0935d41]
On the latest completed run (`track-d1-live-agent-auto-refresh-15c1fc6a7155`), the winning model is **`ensemble_mean`** with a pooled WAPE of **0.0853** (approx. 8.53%) and a pooled MAE of **10.76** across 6 series.
Generated SQL:
WITH latest_run AS (
  SELECT run_id FROM `statmike-scale-forecasting.scale_forecasting.v_run_summary`
  ORDER BY IF(status = 'COMPLETED', 0, 1), created_at DESC LIMIT 1
)
SELECT c.run_id, c.model_type, c.n_series, c.pooled_wape, c.pooled_mae, c.first_forecast_date, c.last_forecast_date, c.refit_modes
FROM `statmike-scale-forecasting.scale_forecasting.v_model_leaderboard_comparable` AS c
JOIN latest_run USING (run_id)
ORDER BY c.pooled_wape ASC NULLS LAST;



[Turn 2 | FVA Waterfall follow-up | rows=8]
The Forecast Value Added (FVA) waterfall results for run `track-d1-live-agent-auto-refresh-15c1fc6a7155` show performance progression across different modeling stages compared to the baseline `arima_plus` model (WAPE 0.0950). The `ensemble_mean` model achieves the highest relative lift of 10.28% with the lowest pooled WAPE of 0.0853.


,stage_name,fva_wape_delta,baseline_wape,pooled_wape,fva_relative_lift_pct,model_type,baseline_model,stage_order
0,2_statistical,0.002632,0.095032,0.092400,2.769413,arima_plus@xreg,arima_plus,2
1,2_statistical,0.000000,0.095032,0.095032,0.000000,arima_plus,arima_plus,2
2,3_univariate_ml_dl,0.006386,0.095032,0.088645,6.720186,timesfm@univariate,arima_plus,3
3,4_covariate_ml_dl,0.003099,0.095032,0.091933,3.260706,timesfm,arima_plus,4
4,5_ensemble,0.009768,0.095032,0.085264,10.278194,ensemble_mean,arima_plus,5
5,5_ensemble,0.009241,0.095032,0.085790,9.724497,ensemble_inverse_error,arima_plus,5



[Turn 3 | Horizon Decay follow-up | rows=14]
The `pooled_wape` for the winning model (`ensemble_mean`) on run `track-d1-live-agent-auto-refresh-15c1fc6a7155` is **0.0623** at horizon step 1, **0.0866** at horizon step 7, and **0.0671** at horizon step 14.



[Turn 4 | Bias & Exception Triage follow-up | rows=5]
There are 5 series flagged with `is_exception = TRUE` for the winning ensemble_mean model on run `track-d1-live-agent-auto-refresh-15c1fc6a7155`. Three series are flagged for chronic under-forecasting and two for chronic over-forecasting.



[Historical Run Pinning | run_id=track-p-p2-live-sim-hierarchy-3mode-01dd2c1edab7]
For run `track-p-p2-live-sim-hierarchy-3mode-01dd2c1edab7` in `v_model_leaderboard_comparable`, the winning model is **`lightgbm`** (tied with `lightgbm@similarity_global`) with a **pooled WAPE** of **0.0933** (~9.33%) and a pooled MAE of **22.55**.



[Cross-Run S&OP Comparison | run_id=None | rows=5]
Here are the 5 most recent completed forecast runs from `v_executive_sop_summary` along with their champion model, holdout-fold pooled WAPE, and total forecasted volume:


,created_at,champion_pooled_wape,status,champion_model,run_id,total_forecast_units
0,2026-10-29 13:00:02,0.085264,COMPLETED,ensemble_mean,track-d1-live-agent-auto-refresh-15c1fc6a7155,10683.148534
1,2026-10-24 16:01:25,0.093330,COMPLETED,lightgbm@similarity_global,track-p-p2-live-sim-hierarchy-3mode-01dd2c1edab7,55311.330490
2,2026-10-24 15:51:09,0.138792,COMPLETED,holtwinters,track-p-p2-live-sharded-sim-global-0916c3ff272e,16829.652332
3,2026-10-23 20:24:13,0.093330,COMPLETED,lightgbm@similarity_global,track-p-live-cluster-pooled-bq-ba4ec9626adb,55311.330490
4,2026-10-23 20:21:49,0.093330,COMPLETED,lightgbm,track-p-live-gce-bq-similarity-94eb9331fddf,55311.330490


## Act 5 · Declarative Post-Run Refresh (`output.ensure_bq_agent`), Offline Parity & MCP Server

Three additional integration paths make conversational analytics seamless across local development, automated DAGs, and external AI coding assistants:
1. **Post-Run Auto-Refresh (`output.ensure_bq_agent = True`):** When enabled on `RunConfig`, `main.run()` automatically refreshes the `DataAgent`'s discovered `source_table` scope after a successful BigQuery run.
2. **Zero-Credential Offline Router (`f.ask(...)` on local runs):** Returns the identical `BQAgentResponse` structure (`mode="offline"`) paired with the matched BigQuery Golden SQL template.
3. **Built-in MCP Tool (`ask_forecast_registry`):** Exposes `mode="probe"`, `mode="auto"`, `mode="bigquery_agent"`, and `mode="offline"` over Model Context Protocol (`python -m scale_forecasting.mcp`).

In [6]:
from scale_forecasting import load_benchmark
from scale_forecasting.mcp import McpServer

# 1. Declarative RunConfig opt-in (`output.ensure_bq_agent = True`):
auto_refresh_cfg = RunConfig(
    run_name="nightly_sop_with_agent_refresh",
    data={"source_table": "source_series_100", "horizon": 14},
    models=["arima_plus", "timesfm"],
    backtest={"enabled": True, "n_folds": 2, "horizon": 14, "step": 14},
    output={"ensure_bq_agent": True},
)
print(
    f"Validated RunConfig '{auto_refresh_cfg.run_name}' | "
    f"output.ensure_bq_agent={auto_refresh_cfg.output.ensure_bq_agent}"
)

# 2. Pure-offline Forecaster.auto(..., where='local').ask(...) parity (zero GCP calls):
ds_local = load_benchmark("dgp_01_smooth_seasonal", n_series=3)
f_local = Forecaster.auto(ds_local.panel_df, tier="first_light", where="local", horizon=7)
offline_ans = f_local.ask("Which model won and what was the FVA waterfall lift?")
print(f"\n[Offline Router | mode={offline_ans.mode}] {offline_ans.answer}")

# 3. Built-in MCP Server (`ask_forecast_registry` with mode='probe' and mode='auto'):
mcp = McpServer()
mcp_probe = json.loads(
    mcp.call_tool(
        "ask_forecast_registry",
        {
            "mode": "probe",
            "project_id": settings.project_id,
            "dataset_id": settings.registry_dataset_id,
        },
    )["content"][0]["text"]
)
print(f"\n[MCP Probe] ready={mcp_probe['ready']} | agent_id={mcp_probe['agent_id']}")

if agent_status.ready:
    mcp_live = json.loads(
        mcp.call_tool(
            "ask_forecast_registry",
            {
                "mode": "auto",
                "project_id": settings.project_id,
                "dataset_id": settings.registry_dataset_id,
                "run_id": latest_completed_run,
                "question": "Return the champion_model and total_forecast_units from v_executive_sop_summary for this run.",
            },
        )["content"][0]["text"]
    )
    print(f"[MCP Auto -> {mcp_live['mode']}] {mcp_live['answer']}")

Validated RunConfig 'nightly_sop_with_agent_refresh' | output.ensure_bq_agent=True



[Offline Router | mode=offline] Computed Forecast Value Added (FVA) across 5 model stage(s) for `auto-first-light-922ba1f811f6`.



[MCP Probe] ready=True | agent_id=sf-registry-scale-forecasting


[MCP Auto -> bigquery_agent] For run `track-d1-live-agent-auto-refresh-15c1fc6a7155` (created at `2026-10-29 13:00:02 UTC`), the champion model is **`ensemble_mean`** with total forecasted volume of **10,683.15** units.


<!-- sf-closing -->
---

## Where next

- **Day-2 Operations & All 5 Launch Pathways:** [`10_registry_operations_and_scale.ipynb`](./10_registry_operations_and_scale.ipynb) — `reg.doctor()`, repair/settle/cancel verbs, Cloud Composer 3 `airflow_tasks`, and 100k-series benchmarks.
- **Glass-Box Auto-Forecaster & Bounded `autofocus`:** [`11_auto_forecaster_and_autofocus.ipynb`](./11_auto_forecaster_and_autofocus.ipynb) — compile transparent `RunConfig` blueprints across 4 exposure tiers.
- **Documentation & Live Q&A Transcript Gallery:** [AI Agents, Skills & MCP Guide](https://statmike.github.io/scale-forecasting/agent_and_mcp_guide/) · [Output schemas & 11 SQL views](https://statmike.github.io/scale-forecasting/output_schemas/) · [Operations runbook](https://statmike.github.io/scale-forecasting/operations/)